In [ ]:
"""SESSION 23 — Cascade Mask R-CNN Swin-B + DCNv2, five-fold CV.
"""

from __future__ import annotations

# %% 1. Imports and dependency bootstrap
import argparse
import csv
import hashlib
import json
import math
import multiprocessing
import os
import random  
import re
import shutil
import signal
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

SESSION_NAME = "session23_cascade_maskrcnn_swin_b_dcn_cv_v10"
RUNTIME_BUILD_ID = "SESSION23-V11-SKIN19-DYNAMIC-MASK-HEAD"
EXPECTED_CLASSES: list[str] = []
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MMCV_WHEEL = (
    "https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/"
    "mmcv-2.1.0-cp311-cp311-manylinux1_x86_64.whl"
)
TORCH_INDEX = "https://download.pytorch.org/whl/cu118"
ENV_ROOT = Path("/tmp/session23_openmmlab")
ISOLATED_ENV = ENV_ROOT / "venv"
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
KAGGLE_RUNTIME_SCRIPT = Path("/kaggle/working/SESSION_23_runtime.py")
SWIN_B_CASCADE_URL = (
    "https://github.com/SwinTransformer/storage/releases/download/v1.0.2/"
    "cascade_mask_rcnn_swin_base_patch4_window7.pth"
)
AUGMENT_PREFIX = re.compile(
    r"^(?:Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25))"
)


def _headless_environment() -> dict[str, str]:
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    env["MPLCONFIGDIR"] = str(ENV_ROOT / "matplotlib")
    env["PYTHONNOUSERSITE"] = "1"
    env["UV_PYTHON_INSTALL_DIR"] = str(ENV_ROOT / "python")
    env["UV_CACHE_DIR"] = str(ENV_ROOT / "cache")
    env["UV_PYTHON_BIN_DIR"] = str(ENV_ROOT / "bin")
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    return env


DEPENDENCY_PROBE_CODE = r'''import sys
import numpy as np
import cv2, scipy, torch, torchvision, mmcv, mmengine, mmdet
from mmcv.ops import ModulatedDeformConv2dPack, nms, roi_align
from mmdet.models import CascadeRCNN
assert sys.version_info[:2] == (3, 11), sys.version
assert torch.cuda.is_available(), 'CUDA is unavailable'
assert torch.__version__.startswith('2.1.0'), torch.__version__
assert torchvision.__version__.startswith('0.16.0'), torchvision.__version__
assert torch.version.cuda == '11.8', torch.version.cuda
assert np.__version__ == '1.26.4', np.__version__
assert scipy.__version__ == '1.14.1', scipy.__version__
assert cv2.__version__ == '4.10.0', cv2.__version__
assert mmcv.__version__ == '2.1.0', mmcv.__version__
assert mmengine.__version__ == '0.10.7', mmengine.__version__
assert mmdet.__version__ == '3.3.0', mmdet.__version__
from scipy.optimize import linear_sum_assignment
linear_sum_assignment(np.array([[1., 2.], [2., 1.]]))
cv2.resize(np.zeros((4, 4, 3), dtype=np.uint8), (8, 8))
layer = ModulatedDeformConv2dPack(8, 8, 3, padding=1).cuda().train()
x = torch.randn(1, 8, 8, 8, device='cuda', requires_grad=True)
y = layer(x)
y.sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all()
sample = torch.randn(1, 4, 16, 16, device='cuda', requires_grad=True)
rois = torch.tensor([[0., 0., 0., 12., 12.]], device='cuda')
roi_align(sample, rois, (7, 7), 1., 0, 'avg', True).sum().backward()
assert sample.grad is not None and torch.isfinite(sample.grad).all()
boxes = torch.tensor([[0., 0., 4., 4.], [1., 1., 5., 5.]], device='cuda')
scores = torch.tensor([0.9, 0.8], device='cuda')
keep = nms(boxes, scores, 0.5)[1]
print('numpy=', np.__version__, 'scipy=', scipy.__version__, 'opencv=', cv2.__version__,
      'torch=', torch.__version__, 'torchvision=', torchvision.__version__,
      'torch_cuda=', torch.version.cuda, 'gpu_count=', torch.cuda.device_count(),
      'mmcv=', mmcv.__version__, 'mmengine=', mmengine.__version__,
      'mmdet=', mmdet.__version__, 'cuda_dcn=', tuple(y.shape),
      'cuda_nms=', int(keep.numel()))'''


def _probe_python(python: Path | str) -> tuple[bool, str]:
    # Importing the symbols is insufficient: an MMCV wheel can import while
    # its compiled CUDA operators are incompatible with Torch/CUDA.  Execute
    # both DCNv2 and NMS on the GPU before any dataset or checkpoint work.
    try:
        result = subprocess.run(
            [str(python), "-c", DEPENDENCY_PROBE_CODE], text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            env=_headless_environment(), timeout=120,
        )
    except (OSError, subprocess.TimeoutExpired) as error:
        return False, repr(error)
    return result.returncode == 0, result.stdout.strip()


def _run_setup(command: list[Path | str], timeout: int = 600) -> None:
    printable = [str(value) for value in command]
    print("+", " ".join(printable), flush=True)
    subprocess.check_call(printable, env=_headless_environment(), timeout=timeout)


def _install_isolated_environment() -> None:
    """Build a pinned environment without replacing Kaggle kernel packages."""
    working = Path("/kaggle/working")
    if not working.is_dir():
        raise RuntimeError(
            "The required OpenMMLab environment is absent. Run this file on "
            "Kaggle with a GPU and Internet enabled."
        )
    nvidia_smi = shutil.which("nvidia-smi")
    if nvidia_smi is None:
        raise RuntimeError(
            "No NVIDIA GPU is visible. In Kaggle Settings, select a GPU "
            "accelerator (T4 x2 is recommended) and restart the session."
        )
    gpu_check = subprocess.run(
        [nvidia_smi, "--query-gpu=name", "--format=csv,noheader"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if gpu_check.returncode or not gpu_check.stdout.strip():
        raise RuntimeError(
            "Kaggle's NVIDIA driver is unavailable. Restart with a GPU "
            "accelerator. nvidia-smi output:\n" + gpu_check.stdout.strip()
        )
    print("  [gpu] visible before setup:", gpu_check.stdout.strip(), flush=True)

    if shutil.disk_usage("/tmp").free < 9 * 1024**3:
        raise RuntimeError("Need 9 GiB free under /tmp for the isolated OpenMMLab environment")
    print("  [deps] creating Python 3.11 OpenMMLab environment under /tmp", flush=True)
    ENV_ROOT.mkdir(parents=True, exist_ok=True)
    if ISOLATED_ENV.exists():
        shutil.rmtree(ISOLATED_ENV)
    installer = ENV_ROOT / "installer"
    if installer.exists():
        shutil.rmtree(installer)
    _run_setup([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                "--target", installer, "uv==0.8.22"], timeout=300)
    uv = installer / "bin" / "uv"
    _run_setup([uv, "venv", "--python", "3.11", "--seed", ISOLATED_ENV], timeout=420)
    pip = [ISOLATED_PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir"]
    _run_setup(pip + ["pip==25.2", "setuptools==75.1.0", "wheel==0.44.0"], timeout=300)
    _run_setup(pip + ["torch==2.1.0", "torchvision==0.16.0",
                      "--index-url", TORCH_INDEX], timeout=1500)
    _run_setup(pip + [
        "numpy==1.26.4", "scipy==1.14.1", "opencv-python==4.10.0.84",
        "matplotlib==3.9.2", "Pillow==10.4.0", "PyYAML==6.0.2",
        "pycocotools==2.0.10", "shapely==2.0.6", "six==1.16.0",
        "terminaltables==3.1.10", "tqdm==4.66.5", "addict==2.4.0",
        "termcolor==2.4.0", "packaging==24.1", "rich==13.9.4",
        "yapf==0.40.2", "mmengine==0.10.7",
    ], timeout=600)
    _run_setup(pip + ["--no-deps", MMCV_WHEEL, "mmdet==3.3.0"], timeout=600)
    _run_setup([ISOLATED_PYTHON, "-m", "pip", "check"], timeout=60)


def _runtime_source_text() -> str:
    """Recover this source from a file or a directly pasted notebook cell."""
    candidates = []
    module_file = globals().get("__file__")
    if module_file:
        candidates.append(Path(module_file))
    candidates.append(KAGGLE_RUNTIME_SCRIPT)
    for candidate in candidates:
        try:
            text = candidate.read_text(encoding="utf-8")
        except (OSError, UnicodeError):
            continue
        if RUNTIME_BUILD_ID in text and "def main(" in text:
            return text

    try:
        shell = get_ipython()  # type: ignore[name-defined]
        history = shell.user_ns.get("In", [])
    except Exception:
        history = []
    for text in reversed(history):
        if (isinstance(text, str) and RUNTIME_BUILD_ID in text
                and "def main(" in text):
            return text
    raise RuntimeError(
        "Could not recover the Session 23 source for isolated relaunch. "
        "Upload SESSION_23_KAGGLE_RUN_THIS.ipynb instead of a partial cell."
    )


def _write_runtime_source() -> Path:
    source = _runtime_source_text()
    compile(source, str(KAGGLE_RUNTIME_SCRIPT), "exec")
    KAGGLE_RUNTIME_SCRIPT.write_text(source, encoding="utf-8")
    return KAGGLE_RUNTIME_SCRIPT


def _normalized_cli(args: argparse.Namespace) -> list[str]:
    values = [
        "--data-root", args.data_root,
        "--device", args.device,
        "--folds", args.folds,
        "--max-iters", str(args.max_iters),
        "--image-size", str(args.image_size),
        "--accumulation", str(args.accumulation),
        "--seed", str(args.seed),
        "--hours", str(args.hours),
    ]
    if args.work_dir:
        values += ["--work-dir", args.work_dir]
    if args.quick:
        values.append("--quick")
    if not args.skip_existing:
        values.append("--no-skip-existing")
    if args.resume:
        values.append("--resume")
    if args.skip_train:
        values.append("--skip-train")
    return values


def _ensure_dependencies(args: argparse.Namespace) -> int | None:
    """Probe current Python or bootstrap and relaunch in the isolated one."""
    ok, detail = _probe_python(sys.executable)
    if ok:
        print(f"  [deps] isolated environment passed:\n{detail}")
        return None

    print("  [deps] current Python is not usable; automatic bootstrap starting")
    print("  [deps] current probe output:\n" + detail)
    isolated_ok, isolated_detail = _probe_python(ISOLATED_PYTHON)
    if not isolated_ok:
        _install_isolated_environment()
        isolated_ok, isolated_detail = _probe_python(ISOLATED_PYTHON)
    if not isolated_ok:
        raise RuntimeError(
            "Isolated dependency/CUDA probe failed after installation.\n"
            + isolated_detail
        )

    runtime = _write_runtime_source()
    command = [
        str(ISOLATED_PYTHON), "-u", str(runtime), *_normalized_cli(args)
    ]
    print("  [deps] isolated dependency/CUDA probe passed:\n" + isolated_detail)
    print("  [deps] relaunching Session 23 in the isolated Python", flush=True)
    print("+", " ".join(command), flush=True)
    return subprocess.call(command, env={**_headless_environment(), "PYTHONUNBUFFERED": "1"})



# %% 2. Rebuilt dataset and fold-manifest validation
def _split_dirs(root: Path, split: str) -> tuple[Path, Path]:
    """Support both Session-00 and group-disjoint V2 directory layouts."""
    legacy = (root / split / "images", root / split / "labels")
    v2 = (root / "images" / split, root / "labels" / split)
    if all(path.is_dir() for path in v2):
        return v2
    if all(path.is_dir() for path in legacy):
        return legacy
    return v2


def _valid_dataset_root(root: Path) -> bool:
    return (root / "data.yaml").is_file() and all(
        all(path.is_dir() for path in _split_dirs(root, split))
        for split in ("train", "valid", "test")
    )


def _dataset_score(root: Path) -> tuple[int, int, int, int]:
    """Prefer the requested dataset, then reviewed group metadata/layout."""
    counts = tuple(sum(path.suffix.lower() in IMAGE_SUFFIXES
                       for path in _split_dirs(root, split)[0].iterdir())
                   for split in ("train", "valid", "test"))
    return (
        int(counts == (6057, 1085, 1030)),
        int(all((root / "cv" / f"fold_{fold}" / "train.txt").is_file()
                and (root / "cv" / f"fold_{fold}" / "valid.txt").is_file()
                for fold in range(5))),
        int((root / "metadata" / "manifest.csv").is_file()),
        int((root / "images" / "train").is_dir()),
    )



def _find_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if _valid_dataset_root(explicit):
            print(f"  [data] explicit rebuilt dataset: {explicit}")
            return explicit
        raise FileNotFoundError(
            f"--data-root is not a complete rebuilt five-fold dataset: {explicit}"
        )

    candidates: set[Path] = set()
    search_roots = [Path("/kaggle/input")]
    if not search_roots[0].is_dir():
        search_roots.append(Path.cwd())
    for search_root in search_roots:
        if search_root.is_dir():
            for data_yaml in sorted(search_root.rglob("data.yaml")):
                root = data_yaml.parent.resolve()
                if _valid_dataset_root(root):
                    candidates.add(root)
    for candidate in (
        Path("/kaggle/working/Image_Annotation_rebuilt"),
        Path("Image_Annotation_rebuilt"),
        Path("../Image_Annotation_rebuilt"),
    ):
        if _valid_dataset_root(candidate):
            candidates.add(candidate.resolve())
    if not candidates:
        raise FileNotFoundError(
            "No complete rebuilt dataset was found. Attach the Session-00/V2 "
            "dataset or set DATA_ROOT to its exact directory."
        )
    ranked = sorted(candidates, key=lambda path: (_dataset_score(path), str(path)),
                    reverse=True)
    best_score = _dataset_score(ranked[0])
    tied = [path for path in ranked if _dataset_score(path) == best_score]
    if len(tied) > 1:
        choices = "\n".join(f"    - {path}" for path in tied)
        raise RuntimeError(
            "Multiple equally suitable rebuilt datasets were found. Set "
            f"DATA_ROOT explicitly:\n{choices}"
        )
    print(f"  [data] selected rebuilt dataset: {ranked[0]}")
    print(f"  [data] integrity features (cv, manifest, V2 layout)={best_score}")
    return ranked[0]


def _load_class_names(data_yaml: Path) -> list[str]:
    try:
        import yaml
        document = yaml.safe_load(data_yaml.read_text()) or {}
    except ModuleNotFoundError:
        import ast

        lines = data_yaml.read_text(encoding="utf-8").splitlines()
        names_line = next((line.split(":", 1)[1].strip()
                           for line in lines if line.lstrip().startswith("names:")), "")
        count_line = next((line.split(":", 1)[1].strip()
                           for line in lines if line.lstrip().startswith("nc:")), "")
        document = {
            "names": ast.literal_eval(names_line) if names_line else None,
            "nc": int(count_line) if count_line else None,
        }
    names = document.get("names")
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda value: int(value))]
    names = [str(name) for name in names] if isinstance(names, (list, tuple)) else []
    if not names:
        raise ValueError(f"{data_yaml} must contain a non-empty names list or mapping")
    declared_count = document.get("nc")
    if declared_count is not None and int(declared_count) != len(names):
        raise ValueError(
            f"{data_yaml} declares nc={declared_count} but contains {len(names)} names"
        )
    global EXPECTED_CLASSES
    EXPECTED_CLASSES = names
    print(f"  [data] loaded {len(names)} classes from {data_yaml}: {names}")
    return names


# %% 2b. On-the-fly stratified 5-fold CV (no pre-built manifests needed)
def _gather_split(root: Path, split: str) -> list[dict]:
    """Collect all labelled images from one dataset split directory."""
    images_dir, labels_dir = _split_dirs(root, split)
    if not images_dir.is_dir():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    items = []
    for image in sorted(images_dir.iterdir()):
        if image.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        label = labels_dir / f"{image.stem}.txt"
        if not label.is_file():
            raise FileNotFoundError(f"Missing label for {image}: {label}")
        key = str(image.resolve())
        items.append({"image": image.resolve(), "label": label.resolve(), "key": key})
    if not items:
        raise ValueError(f"No supported images found in {images_dir}")
    return items


def _assign_inferred_groups(root: Path, items: list[dict]) -> dict:
    """Join known augmentation families and exact decoded-pixel duplicates."""
    from PIL import Image

    manifest = root / "metadata" / "manifest.csv"
    metadata = {}
    if manifest.is_file():
        with manifest.open(newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                name = Path(row.get("image_file", "")).name
                if not name or name in metadata or not row.get("split_group_id"):
                    raise ValueError("Incomplete or ambiguous V2 group manifest")
                metadata[name] = row["split_group_id"]

    parent = list(range(len(items)))
    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    seen = {}
    pixel_labels = defaultdict(set)
    for index, item in enumerate(items):
        image = item["image"]
        source = AUGMENT_PREFIX.sub("", image.stem.split(".rf.", 1)[0], count=1)
        if metadata:
            if image.name not in metadata:
                raise ValueError(f"Missing V2 group manifest entry: {image.name}")
            source = metadata[image.name]
        with Image.open(image) as handle:
            rgb = handle.convert("RGB")
            pixel = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        item["source_key"] = source
        item["pixel_hash"] = pixel
        pixel_labels[pixel].add(item["label"].read_text().strip())
        for token in (("source", source), ("pixel", pixel)):
            if token in seen:
                parent[find(index)] = find(seen[token])
            else:
                seen[token] = index

    group_sources = defaultdict(set)
    for index, item in enumerate(items):
        group_sources[find(index)].add(item["source_key"])
    for index, item in enumerate(items):
        sources = sorted(group_sources[find(index)])
        item["group_id"] = hashlib.sha256(json.dumps(sources).encode()).hexdigest()[:20]
    return {
        "inferred_or_manifest_source_groups": len(group_sources),
        "exact_pixel_annotation_conflict_sets": sum(len(labels) > 1 for labels in pixel_labels.values()),
        "patient_identity_verified": False,
        "source_group_method": "V2 manifest plus exact pixels" if metadata else
                               "known augmentation filenames plus exact pixels",
    }


def _manifest_lookup(pool: list[dict]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for item in pool:
        for alias in (item["key"], item["image"].name, item["image"].stem):
            if alias in lookup and lookup[alias]["key"] != item["key"]:
                raise RuntimeError(
                    f"Ambiguous image alias {alias}; fold lists cannot be resolved"
                )
            lookup[alias] = item
    return lookup


def _lookup_manifest_item(value: str, lookup: dict[str, dict]) -> dict | None:
    candidate = Path(value)
    return lookup.get(value) or lookup.get(candidate.name) or lookup.get(candidate.stem)


def _read_fold_list(
    path: Path,
    development_lookup: dict[str, dict],
    test_lookup: dict[str, dict],
) -> tuple[list[dict], int, int]:
    resolved: dict[str, dict] = {}
    missing = 0
    excluded_test = 0
    for raw in path.read_text().splitlines():
        value = raw.strip()
        if not value or value.startswith("#"):
            continue
        item = _lookup_manifest_item(value, development_lookup)
        if item is not None:
            resolved[item["key"]] = item
        elif _lookup_manifest_item(value, test_lookup) is not None:
            excluded_test += 1
        else:
            missing += 1
    return [resolved[key] for key in sorted(resolved)], missing, excluded_test


def _verify_group_manifest(root: Path, folds: list[dict], test: list[dict]) -> dict:
    """Prove group isolation only when the V2 metadata manifest is complete."""
    manifest = root / "metadata" / "manifest.csv"
    if not manifest.is_file():
        return {
            "group_manifest_present": False,
            "group_disjoint_verified": False,
            "reason": "metadata/manifest.csv is absent",
        }
    with manifest.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    by_basename = {
        Path(row["image_file"]).name: row
        for row in rows if row.get("image_file")
    }
    required = [item for fold in folds for item in fold["valid"]] + test
    missing_rows = [item["image"].name for item in required
                    if item["image"].name not in by_basename]
    missing_groups = [
        item["image"].name for item in required
        if item["image"].name in by_basename
        and not by_basename[item["image"].name].get("split_group_id", "")
    ]
    if missing_rows or missing_groups:
        raise RuntimeError(
            "V2 group manifest is incomplete: "
            f"missing_rows={len(missing_rows)}, missing_group_ids={len(missing_groups)}"
        )
    test_groups = {
        by_basename[item["image"].name]["split_group_id"] for item in test
    }
    checks = []
    for fold in folds:
        train_groups = {
            by_basename[item["image"].name]["split_group_id"]
            for item in fold["train"]
        }
        valid_groups = {
            by_basename[item["image"].name]["split_group_id"]
            for item in fold["valid"]
        }
        train_valid_overlap = train_groups & valid_groups
        test_overlap = (train_groups | valid_groups) & test_groups
        if train_valid_overlap or test_overlap:
            raise RuntimeError(
                f"Group leakage in fold {fold['fold']}: "
                f"train/valid={len(train_valid_overlap)}, "
                f"development/test={len(test_overlap)}"
            )
        checks.append({
            "fold": fold["fold"],
            "train_valid_group_overlap": 0,
            "development_test_group_overlap": 0,
        })
    return {
        "group_manifest_present": True,
        "group_disjoint_verified": True,
        "fold_checks": checks,
    }


def _load_predefined_folds(
    root: Path, development: list[dict], test: list[dict]
) -> tuple[list[dict], dict] | None:
    folders = [root / "cv" / f"fold_{fold}" for fold in range(5)]
    if not all((folder / "train.txt").is_file()
               and (folder / "valid.txt").is_file() for folder in folders):
        return None
    development_lookup = _manifest_lookup(development)
    test_lookup = _manifest_lookup(test)
    development_keys = {item["key"] for item in development}
    validation_counts: Counter = Counter()
    folds = []
    missing = 0
    excluded_test = 0
    for fold, folder in enumerate(folders):
        train, train_missing, train_test = _read_fold_list(
            folder / "train.txt", development_lookup, test_lookup
        )
        valid, valid_missing, valid_test = _read_fold_list(
            folder / "valid.txt", development_lookup, test_lookup
        )
        missing += train_missing + valid_missing
        excluded_test += train_test + valid_test
        train_keys = {item["key"] for item in train}
        valid_keys = {item["key"] for item in valid}
        if train_keys & valid_keys:
            raise RuntimeError(f"Fold {fold} has train/validation image overlap")
        if train_keys | valid_keys != development_keys:
            raise RuntimeError(
                f"Fold {fold} does not cover every available development image"
            )
        validation_counts.update(valid_keys)
        folds.append({"fold": fold, "train": train, "valid": valid})
    bad = {key: count for key, count in validation_counts.items() if count != 1}
    if set(validation_counts) != development_keys or bad:
        raise RuntimeError(
            "Five validation folds must cover every development image exactly once; "
            f"bad_count={len(bad)}"
        )
    return folds, {
        "cv_mode": "predefined Session-00 five-fold manifests",
        "development_images": len(development),
        "untouched_test_images": len(test),
        "fold_val_sizes": {fold["fold"]: len(fold["valid"]) for fold in folds},
        "test_manifest_references_excluded": excluded_test,
        "missing_manifest_references_filtered": missing,
        **_verify_group_manifest(root, folds, test),
    }


def _multi_hot(items: list[dict], num_classes: int) -> "np.ndarray":
    """Binary [N, C] label matrix for stratified splitting."""
    import numpy as np
    matrix = np.zeros((len(items), num_classes), dtype=np.int32)
    for i, item in enumerate(items):
        for line in item["label"].read_text().splitlines():
            values = line.split()
            if values:
                try:
                    cls = int(values[0])
                    if 0 <= cls < num_classes:
                        matrix[i, cls] = 1
                except ValueError:
                    pass
    return matrix


def _generate_folds(
    trainval: list[dict], num_classes: int, n_splits: int, seed: int,
    test: list[dict] | None = None,
) -> tuple[list[dict], dict]:
    """Assign whole inferred source/pixel groups to validation folds."""
    import numpy as np
    groups = defaultdict(list)
    for index, item in enumerate(trainval):
        groups[item["group_id"]].append(index)
    if len(groups) < n_splits:
        raise ValueError("Five-fold CV requires at least five independent source groups")
    group_ids = sorted(groups)
    Y = _multi_hot(trainval, num_classes)
    features = np.array([
        [int(Y[groups[group], cls].sum()) for cls in range(num_classes)]
        + [len(groups[group])]
        for group in group_ids
    ], dtype=float)
    target = np.maximum(features.sum(axis=0) / n_splits, 1.)
    best = None
    for trial in range(32):
        rng = random.Random(seed + trial)
        order = list(range(len(group_ids)))
        rng.shuffle(order)
        order.sort(key=lambda index: -float((features[index] / target).max()))
        counts = np.zeros((n_splits, num_classes + 1))
        assignment = {}
        for index in order:
            scores = []
            for fold in range(n_splits):
                after = counts.copy()
                after[fold] += features[index]
                scores.append(float(((after / target) ** 2).sum()))
            minimum = min(scores)
            choices = [fold for fold, score in enumerate(scores)
                       if abs(score - minimum) < 1e-10]
            chosen = rng.choice(choices)
            counts[chosen] += features[index]
            assignment[group_ids[index]] = chosen
        objective = int((counts[:, :num_classes] == 0).sum()) * 1e6 + float(
            (((counts - target) / target) ** 2).sum())
        if best is None or objective < best[0]:
            best = (objective, assignment)
    assignment = best[1]
    folds = []
    for f in range(n_splits):
        valid = [item for item in trainval if assignment[item["group_id"]] == f]
        train = [item for item in trainval if assignment[item["group_id"]] != f]
        if not valid or not train:
            raise ValueError(f"Fold {f} is empty after group assignment")
        if {item["group_id"] for item in valid} & {item["group_id"] for item in train}:
            raise AssertionError("Source group crossed train/validation")
        if {item["pixel_hash"] for item in valid} & {item["pixel_hash"] for item in train}:
            raise AssertionError("Exact decoded pixel copy crossed train/validation")
        for subset in (train, valid):
            subset_classes = set().union(*(_item_classes(item) for item in subset))
            if subset_classes != set(range(num_classes)):
                raise ValueError(f"Fold {f} lacks a class after group assignment")
        folds.append({
            "fold": f, "train": train, "valid": valid,
        })
    assert sorted(item["key"] for fold in folds for item in fold["valid"]) == sorted(
        item["key"] for item in trainval)
    val_sizes = {fold["fold"]: len(fold["valid"]) for fold in folds}
    test_groups = {item["group_id"] for item in (test or [])}
    development_groups = set(groups)
    print(
        f"  [cv] generated {n_splits} group-aware folds; "
        f"val sizes: {val_sizes}"
    )
    integrity = {
        "available_images": len(trainval),
        "cv_images": len(trainval),
        "fold_val_sizes": val_sizes,
        "cv_mode": "inferred source and exact-pixel group CV",
        "group_manifest_present": False,
        "group_disjoint_verified": True,
        "patient_disjoint_verified": False,
        "development_test_group_overlap": len(development_groups & test_groups),
        "warning": (
            "Source groups are inferred from known filename augmentations and "
            "exact decoded pixels, not verified patient identities. The original "
            "test partition is excluded, but may not be independent."
        ),
    }
    return folds, integrity




def _item_classes(item: dict) -> set[int]:
    classes = set()
    for line in item["label"].read_text().splitlines():
        values = line.split()
        if values:
            classes.add(int(values[0]))
    return classes


def _balanced_subset(items: list[dict], limit: int, seed: int) -> list[dict]:
    if len(items) <= limit:
        return list(items)
    shuffled = list(items)
    random.Random(seed).shuffle(shuffled)
    chosen = []
    used = set()
    for class_id in range(len(EXPECTED_CLASSES)):
        for item in shuffled:
            if item["key"] not in used and class_id in _item_classes(item):
                chosen.append(item)
                used.add(item["key"])
                break
    for item in shuffled:
        if len(chosen) >= limit:
            break
        if item["key"] not in used:
            chosen.append(item)
            used.add(item["key"])
    return chosen


# %% 3. YOLO polygons to COCO
def _polygon_area(xs: list[float], ys: list[float]) -> float:
    return 0.5 * abs(sum(
        xs[index] * ys[(index + 1) % len(xs)]
        - xs[(index + 1) % len(xs)] * ys[index]
        for index in range(len(xs))
    ))


def _yolo_to_coco(
    items: list[dict],
    names: list[str],
    destination: Path,
    image_ids: dict[str, int],
) -> dict:
    from PIL import Image

    images = []
    annotations = []
    annotation_id = 1
    skipped = 0
    class_instances: Counter = Counter()
    for item in sorted(items, key=lambda value: image_ids[value["key"]]):
        image_id = image_ids[item["key"]]
        with Image.open(item["image"]) as handle:
            width, height = handle.size
        images.append({
            "id": image_id,
            "file_name": str(item["image"]),
            "width": width,
            "height": height,
        })
        for line_number, line in enumerate(item["label"].read_text().splitlines(), 1):
            values = line.strip().split()
            if not values:
                continue
            if len(values) < 7 or (len(values) - 1) % 2:
                skipped += 1
                continue
            try:
                class_id = int(values[0])
                coordinates = [float(value) for value in values[1:]]
            except ValueError:
                skipped += 1
                continue
            if not all(math.isfinite(value) for value in coordinates):
                skipped += 1
                continue
            if not 0 <= class_id < len(names):
                raise ValueError(
                    f"Invalid class {class_id} in {item['label']}:{line_number}"
                )
            xs = [max(0.0, min(1.0, coordinates[i])) * width
                  for i in range(0, len(coordinates), 2)]
            ys = [max(0.0, min(1.0, coordinates[i])) * height
                  for i in range(1, len(coordinates), 2)]
            area = _polygon_area(xs, ys)
            x0, x1 = min(xs), max(xs)
            y0, y1 = min(ys), max(ys)
            if not math.isfinite(area) or area <= 1.0 or x1 <= x0 or y1 <= y0:
                skipped += 1
                continue
            polygon = [coordinate for pair in zip(xs, ys) for coordinate in pair]
            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": class_id + 1,
                "segmentation": [polygon],
                "area": float(area),
                "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)],
                "iscrowd": 0,
            })
            annotation_id += 1
            class_instances[class_id] += 1
    document = {
        "info": {
            "description": "Session 23 YOLO-to-COCO fold export",
            "version": "1.0",
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": index + 1, "name": name, "supercategory": "skin"}
            for index, name in enumerate(names)
        ],
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(document))
    print(
        f"  [coco] {destination.name}: {len(images)} images, "
        f"{len(annotations)} instances, invalid skipped={skipped}"
    )
    return {
        "images": len(images),
        "annotations": len(annotations),
        "skipped_polygons": skipped,
        "instances_per_class": {
            names[index]: int(class_instances[index]) for index in range(len(names))
        },
    }


# %% 4. Official Swin-B Cascade checkpoint conversion
def _download_and_prepare_checkpoint(
    work: Path, config: Path
) -> tuple[Path, dict]:
    """Keep only official checkpoint tensors that exactly fit this model."""
    prepared = work / "pretrained" / "cascade_swin_b_exactshape_v9.pth"
    report_path = work / "pretrained" / "checkpoint_adaptation.json"
    if (prepared.is_file() and prepared.stat().st_size > 1_000_000
            and report_path.is_file()):
        print(f"  [weights] prepared checkpoint exists: {prepared}")
        return prepared, json.loads(report_path.read_text())
    raw = work / "pretrained" / "cascade_swin_b_coco_raw.pth"
    raw.parent.mkdir(parents=True, exist_ok=True)
    print("  [weights] downloading official Swin-B Cascade Mask R-CNN COCO weights ...")
    try:
        urlretrieve(SWIN_B_CASCADE_URL, raw)
    except Exception as error:
        raw.unlink(missing_ok=True)
        raise RuntimeError(
            "Swin-B checkpoint download failed. Enable Kaggle Internet and rerun."
        ) from error
    if raw.stat().st_size <= 1_000_000:
        raw.unlink(missing_ok=True)
        raise RuntimeError("Downloaded Swin-B checkpoint is unexpectedly small")

    import torch
    from mmengine.config import Config
    from mmdet.registry import MODELS
    from mmdet.models.backbones.swin import swin_converter
    from mmdet.utils import register_all_modules

    checkpoint = torch.load(raw, map_location="cpu", weights_only=False)
    loaded = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))
    original = {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in loaded.items()
    }
    backbone = {
        key[len("backbone."):]: value
        for key, value in original.items()
        if key.startswith("backbone.")
    }
    other = {
        key: value for key, value in original.items()
        if not key.startswith("backbone.")
    }
    if not backbone:
        raise RuntimeError("Downloaded checkpoint contains no Swin backbone weights")
    needs_conversion = any(
        key.startswith("layers.") or "patch_embed.proj" in key
        for key in backbone
    )
    converted_backbone = swin_converter(backbone) if needs_conversion else {
        f"backbone.{key}": value for key, value in backbone.items()
    }
    converted = {**other, **converted_backbone}
    if not any(key.startswith("backbone.stages.") for key in converted):
        raise RuntimeError("Swin checkpoint key conversion did not produce stage weights")

    register_all_modules(init_default_scope=True)
    model = MODELS.build(Config.fromfile(str(config)).model)
    target = model.state_dict()
    retained = {}
    shape_mismatches = []
    unknown = []
    for key, value in converted.items():
        if key not in target:
            unknown.append(key)
        elif tuple(value.shape) != tuple(target[key].shape):
            shape_mismatches.append(key)
        else:
            retained[key] = value
    required = {
        "swin_backbone": any(key.startswith("backbone.stages.") for key in retained),
        "fpn": any(key.startswith("neck.") for key in retained),
        "rpn": any(key.startswith("rpn_head.") for key in retained),
        "cascade_bbox": any(key.startswith("roi_head.bbox_head.2") for key in retained),
    }
    # The DCNv2 mask layers may have parameters absent from a standard-conv
    # checkpoint. Transfer exact-shape mask weights when available, but do not
    # reject otherwise valid backbone/FPN/RPN/cascade initialization for them.
    mask_feature_transfer = {
        str(stage): any(key.startswith(f"roi_head.mask_head.{stage}.convs.")
                        for key in retained)
        for stage in range(3)
    }
    if not all(required.values()) or len(retained) < 100:
        raise RuntimeError(
            "Official checkpoint is not compatible with the requested Swin-B "
            f"Cascade architecture: required={required}, retained={len(retained)}"
        )
    torch.save({
        "state_dict": retained,
        "meta": {
            "source": SWIN_B_CASCADE_URL,
            "adaptation": "Only exact-name, exact-shape tensors were retained.",
        },
    }, prepared)
    raw.unlink(missing_ok=True)
    report = {
        "source_url": SWIN_B_CASCADE_URL,
        "converted_swin_keys": needs_conversion,
        "retained_tensors": len(retained),
        "shape_mismatch_tensors": shape_mismatches,
        "unknown_tensors": len(unknown),
        "required_components_present": required,
        "mask_features_transferred_by_stage": mask_feature_transfer,
    }
    report_path.write_text(json.dumps(report, indent=2))
    print(
        f"  [weights] converted_swin_keys={needs_conversion}; "
        f"retained={len(retained)}; shape_mismatches={len(shape_mismatches)}; "
        f"unknown={len(unknown)}"
    )
    return prepared, report


# %% 5. MMDetection configuration and launch helpers
def _mmdet_tool(name: str) -> str:
    import mmdet

    root = Path(mmdet.__file__).resolve().parent
    candidates = [
        root / ".mim" / "tools" / f"{name}.py",
        root / "tools" / f"{name}.py",
        root.parent / "tools" / f"{name}.py",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(f"MMDetection tools/{name}.py was not found under {root}")


def _detect_gpu_ids(device: str) -> list[str]:
    if device != "auto":
        identifiers = [part.strip() for part in device.split(",") if part.strip()]
        if device == "cpu" or not identifiers:
            raise RuntimeError("Session 23 requires a CUDA GPU")
        return identifiers
    count = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
        text=True,
    ).strip())
    if count < 1:
        raise RuntimeError("No CUDA GPU detected. Enable a Kaggle GPU accelerator.")
    return [str(index) for index in range(min(2, count))]


def _bbox_head(num_classes: int, target_stds: tuple[float, ...]) -> str:
    return f"""dict(
            type='ConvFCBBoxHead', num_shared_convs=4, num_shared_fcs=1,
            in_channels=256, conv_out_channels=256, fc_out_channels=1024,
            roi_feat_size=7, num_classes={num_classes},
            reg_class_agnostic=False, reg_decoded_bbox=True,
            norm_cfg=dict(type='GN', num_groups=32, requires_grad=True),
            bbox_coder=dict(
                type='DeltaXYWHBBoxCoder', target_means=[0., 0., 0., 0.],
                target_stds={list(target_stds)!r}),
            loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False,
                          loss_weight=1.0),
            loss_bbox=dict(type='GIoULoss', loss_weight=10.0))"""


def _write_training_config(
    destination: Path,
    train_json: Path,
    valid_json: Path,
    work_dir: Path,
    checkpoint: Path,
    names: list[str],
    gpu_count: int,
    max_iters: int,
    image_size: int,
    accumulation: int,
    seed: int,
) -> None:
    classes = repr(tuple(names))
    palette = repr([(220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230)])
    interval = min(max_iters, max(500, max_iters // 2))
    warmup = min(500, max(10, max_iters // 10))
    effective_batch = gpu_count * accumulation
    learning_rate = 1e-4 * effective_batch / 16.0
    milestone_1 = max(warmup + 1, int(max_iters * 0.80))
    milestone_2 = max(milestone_1 + 1, int(max_iters * 0.92))
    bbox_heads = ",\n        ".join([
        _bbox_head(len(names), (0.1, 0.1, 0.2, 0.2)),
        _bbox_head(len(names), (0.05, 0.05, 0.1, 0.1)),
        _bbox_head(len(names), (0.033, 0.033, 0.067, 0.067)),
    ])
    config = f'''# Auto-generated by SESSION 23.
_base_ = ['mmdet::cascade_rcnn/cascade-mask-rcnn_r50_fpn_1x_coco.py']

default_scope = 'mmdet'
metainfo = dict(classes={classes}, palette={palette})
image_size = ({image_size}, {image_size})

data_preprocessor = dict(
    type='DetDataPreprocessor',
    mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375],
    bgr_to_rgb=True, pad_mask=True, mask_pad_value=0, pad_size_divisor=32,
)
model = dict(
    data_preprocessor=data_preprocessor,
    backbone=dict(
        _delete_=True, type='SwinTransformer', pretrain_img_size=224,
        embed_dims=128, depths=[2, 2, 18, 2], num_heads=[4, 8, 16, 32],
        window_size=7, mlp_ratio=4, qkv_bias=True, qk_scale=None,
        drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.3,
        patch_norm=True, out_indices=(0, 1, 2, 3), with_cp=True,
        convert_weights=False, frozen_stages=-1, init_cfg=None,
    ),
    neck=dict(in_channels=[128, 256, 512, 1024]),
    roi_head=dict(
        bbox_head=[
        {bbox_heads}
        ],
        mask_head=dict(
            num_classes={len(names)},
            # Each of the three cascade heads has four DCNv2 mask convolutions.
            conv_cfg=dict(type='DCNv2', deform_groups=1),
            norm_cfg=dict(type='GN', num_groups=32, requires_grad=True),
        ),
    ),
)

train_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=False),
    dict(type='RandomFlip', prob=0.5),
    dict(type='RandomResize', scale=image_size, ratio_range=(0.75, 1.25),
         resize_type='Resize', keep_ratio=True),
    dict(type='RandomCrop', crop_size=image_size, crop_type='absolute',
         recompute_bbox=True, allow_negative_crop=False),
    dict(type='FilterAnnotations', min_gt_bbox_wh=(1.0, 1.0), by_mask=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='Resize', scale=image_size, keep_ratio=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs', meta_keys=(
        'img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

# _delete_=True prevents cascade base config's AspectRatioBatchSampler from
# leaking into the merged dict alongside InfiniteSampler -> mutually exclusive crash.
train_dataloader = dict(
    _delete_=True,
    batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset', data_root='',
        ann_file={str(train_json)!r}, data_prefix=dict(img=''),
        metainfo=metainfo, filter_cfg=dict(filter_empty_gt=True, min_size=1),
        pipeline=train_pipeline,
    ),
)
# OOF images remain untouched until final-checkpoint prediction. Selecting the
# best checkpoint on this same fold would bias the reported OOF score upward.
val_dataloader = None
val_evaluator = None
test_dataloader = None
test_evaluator = None

custom_keys = {{
    'backbone': dict(lr_mult=0.1, decay_mult=1.0),
    'absolute_pos_embed': dict(decay_mult=0.0),
    'relative_position_bias_table': dict(decay_mult=0.0),
    'norm': dict(decay_mult=0.0),
}}
optim_wrapper = dict(
    _delete_=True, type='AmpOptimWrapper', loss_scale='dynamic',
    accumulative_counts={accumulation},
    optimizer=dict(type='AdamW', lr={learning_rate}, betas=(0.9, 0.999),
                   weight_decay=0.05),
    paramwise_cfg=dict(custom_keys=custom_keys),
    clip_grad=dict(max_norm=5.0, norm_type=2),
)
param_scheduler = []
custom_imports = dict(imports=['session23_budget_hook'], allow_failed_imports=False)
custom_hooks = [dict(type='Session23BudgetHook', warmup={warmup}, accumulation={accumulation})]
train_cfg = dict(
    _delete_=True, type='IterBasedTrainLoop', max_iters={max_iters},
    val_interval={max_iters + 1},
)
val_cfg = None
test_cfg = None
default_hooks = dict(
    logger=dict(type='LoggerHook', interval=50),
    checkpoint=dict(
        type='CheckpointHook', by_epoch=False, interval={interval},
        save_last=True, max_keep_ckpts=2,
    ),
)
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
randomness = dict(seed={seed}, deterministic=False, diff_rank_seed=True)
env_cfg = dict(
    cudnn_benchmark=True,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)
launcher = 'none'
work_dir = {str(work_dir)!r}
load_from = {str(checkpoint)!r}
resume = False
'''
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(config)


BUDGET_HOOK_SOURCE = '''import json
import math
import os
import time
from pathlib import Path
import torch
from mmengine.hooks import Hook
from mmdet.registry import HOOKS

@HOOKS.register_module()
class Session23BudgetHook(Hook):
    priority = 'NORMAL'
    def __init__(self, warmup=500, accumulation=2):
        self.warmup, self.accumulation = warmup, accumulation
    def before_run(self, runner):
        # Test runners retain a ConfigDict with an `optimizer` field. That
        # field is another configuration, not a built torch optimizer.
        optimizer = getattr(runner.optim_wrapper, 'optimizer', None)
        if isinstance(optimizer, torch.optim.Optimizer):
            self.base_lrs = [g['lr'] for g in optimizer.param_groups]
    def before_train(self, runner):
        if not hasattr(self, 'base_lrs'):
            raise RuntimeError('Session23BudgetHook requires a built optimizer before training')
        self.seconds = float(os.environ.get('SESSION23_TRAIN_SECONDS', '7200'))
        self.started = time.monotonic()
        self.start_fraction = runner.iter / runner.max_iters
        self.stopped = False
    def before_train_iter(self, runner, batch_idx, data_batch=None):
        elapsed = time.monotonic() - self.started
        fraction = max(runner.iter / runner.max_iters,
                       self.start_fraction + (1-self.start_fraction) * elapsed / self.seconds)
        # One clock for both DDP ranks: never stop one rank while the other backpropagates.
        value = torch.tensor(min(fraction, 1.), device=next(runner.model.parameters()).device)
        if torch.distributed.is_initialized():
            torch.distributed.broadcast(value, src=0)
        self.fraction = value.item()
        factor = (0.01 + 0.99 * (1 + math.cos(math.pi*self.fraction))/2)
        factor *= min(1., (runner.iter+1)/max(1, self.warmup))
        for group, base_lr in zip(runner.optim_wrapper.optimizer.param_groups, self.base_lrs):
            group['lr'] = base_lr * factor
    def after_train_iter(self, runner, batch_idx, data_batch=None, outputs=None):
        if self.fraction >= 1. and (runner.iter+1) % self.accumulation == 0:
            self.stopped = True
            runner.train_loop.stop_training = True
        if self.stopped or runner.iter+1 >= runner.max_iters:
            # Save at the iteration boundary so MMEngine's resume metadata is correct.
            runner.save_checkpoint(runner.work_dir, 'budget_final.pth', by_epoch=False,
                                   save_optimizer=True, save_param_scheduler=False)
    def after_train(self, runner):
        if not torch.distributed.is_initialized() or torch.distributed.get_rank() == 0:
            report = dict(iterations=runner.iter, iteration_cap=runner.max_iters,
                training_seconds=time.monotonic()-self.started, allocated_seconds=self.seconds,
                stopped_for_time=self.stopped, reached_iteration_cap=runner.iter >= runner.max_iters)
            (Path(runner.work_dir)/'training_budget_report.json').write_text(json.dumps(report, indent=2))
'''


def _fold_training_budget(remaining_seconds, remaining_folds, evaluation_seconds):
    # Per-fold inference/scoring + checkpoint/process overhead; pooled scoring/export.
    seconds = (remaining_seconds - 1200.) / remaining_folds - evaluation_seconds - 180.
    if seconds < 120:
        raise TimeoutError('Insufficient time for remaining folds and OOF evaluation.')
    return seconds


def _bounded_call(command, environment, seconds):
    if not math.isfinite(seconds) or seconds <= 0:
        raise TimeoutError('No time remains for the requested Session 23 subprocess')
    process = subprocess.Popen(command, env=environment, start_new_session=True)
    try:
        code = process.wait(timeout=seconds)
    except (subprocess.TimeoutExpired, KeyboardInterrupt) as error:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        if isinstance(error, KeyboardInterrupt):
            raise
        raise TimeoutError('Session 23 subprocess exceeded its time allocation')
    if code:
        raise subprocess.CalledProcessError(code, command)


def _write_empty_mask_guard(work: Path) -> Path:
    """Install a runner-local guard for zero-positive DCNv2 mask batches."""
    work.mkdir(parents=True, exist_ok=True)
    (work / "session23_budget_hook.py").write_text(BUDGET_HOOK_SOURCE, encoding="utf-8")
    guard = work / "session23_empty_mask_guard.py"
    guard.write_text('''from mmdet.models.roi_heads.mask_heads.fcn_mask_head import FCNMaskHead


def install():
    if getattr(FCNMaskHead.forward, "_session23_empty_safe", False):
        return
    original_forward = FCNMaskHead.forward

    def forward(self, x):
        if x.shape[0] or not self.conv_cfg or self.conv_cfg.get("type") != "DCNv2":
            return original_forward(self, x)
        # FCNMaskHead.loss_and_target handles empty predictions. MMCV DCNv2
        # cannot process their zero-batch features, so supply the empty shape
        # directly. Connect all parameters with zero gradients for DDP.
        scale = self.scale_factor if self.upsample is not None else 1
        if isinstance(scale, (tuple, list)):
            scale_h, scale_w = scale
        else:
            scale_h = scale_w = scale
        channels = 1 if self.class_agnostic else self.num_classes
        output = x.new_zeros((0, channels, int(x.shape[-2] * scale_h),
                              int(x.shape[-1] * scale_w)))
        zero = x.sum() * 0
        for parameter in self.parameters():
            zero = zero + parameter.sum() * 0
        return output + zero

    forward._session23_empty_safe = True
    FCNMaskHead.forward = forward
''', encoding="utf-8")
    launcher = work / "session23_safe_mmdet_launcher.py"
    launcher.write_text('''import runpy
import sys

from session23_empty_mask_guard import install

install()
tool = sys.argv[1]
sys.argv = [tool, *sys.argv[2:]]
runpy.run_path(tool, run_name="__main__")
''', encoding="utf-8")
    return launcher


def _validate_model_build(config: Path, launcher: Path) -> None:
    """Build once in a fresh process and prove cascade/Swin-B/DCNv2 are real."""
    code = r'''
import sys
from mmengine.config import Config
from mmdet.registry import MODELS
from mmdet.utils import register_all_modules
from mmcv.ops import ModulatedDeformConv2dPack
from session23_empty_mask_guard import install
register_all_modules(init_default_scope=True)
install()
cfg = Config.fromfile(sys.argv[1])
model = MODELS.build(cfg.model)
assert model.__class__.__name__ == 'CascadeRCNN'
assert list(model.backbone.num_features) == [128, 256, 512, 1024]
assert [len(stage.blocks) for stage in model.backbone.stages] == [2, 2, 18, 2]
assert model.roi_head.num_stages == 3
assert len(model.roi_head.mask_head) == 3, 'expected one mask head per cascade stage'
dcn_per_stage = [
    sum(isinstance(module, ModulatedDeformConv2dPack) for module in head.modules())
    for head in model.roi_head.mask_head
]
assert dcn_per_stage == [4, 4, 4], (
    f'expected four DCNv2 mask convolutions per stage, found {dcn_per_stage}'
)
import torch
empty = torch.empty((0, 256, 14, 14), requires_grad=True)
empty_prediction = model.roi_head.mask_head[0](empty)
expected_mask_classes = model.roi_head.mask_head[0].num_classes
assert empty_prediction.shape == (0, expected_mask_classes, 28, 28), empty_prediction.shape
empty_prediction.sum().backward()
assert empty.grad is not None
assert all(param.grad is not None for param in model.roi_head.mask_head[0].parameters())
print('architecture=Genuine CascadeRCNN + Swin-B; cascade_stages=3; '
      f'DCNv2_mask_layers_by_stage={dcn_per_stage}; DCNv2_total={sum(dcn_per_stage)}; '
      'empty_mask_forward_and_backward=passed')
'''
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(launcher.parent)
    subprocess.check_call([sys.executable, "-c", code, str(config)], env=environment)


def _best_checkpoint(work_dir: Path) -> Path | None:
    """Use the final or latest iteration checkpoint, never OOF-selected best."""
    if (work_dir / "budget_final.pth").is_file():
        return work_dir / "budget_final.pth"
    # Restored/copied files need not retain timestamps in iteration order.
    last = sorted((path for path in work_dir.glob("iter_*.pth")
                   if path.stem.removeprefix("iter_").isdigit()),
                  key=lambda path: int(path.stem.removeprefix("iter_")))
    return last[-1] if last else None


def _completed_checkpoint(work_dir: Path, config: Path) -> Path | None:
    marker = work_dir / "SESSION_23_TRAINING_COMPLETE.json"
    if not marker.is_file():
        return None
    try:
        record = json.loads(marker.read_text())
        digest = hashlib.sha256(config.read_bytes()).hexdigest()
        checkpoint = Path(record["checkpoint"])
        if record.get("config_sha256") == digest and checkpoint.is_file():
            return checkpoint
    except (KeyError, OSError, ValueError, json.JSONDecodeError):
        pass
    return None


def _mark_training_complete(work_dir: Path, config: Path, checkpoint: Path) -> None:
    marker = work_dir / "SESSION_23_TRAINING_COMPLETE.json"
    marker.write_text(json.dumps({
        "checkpoint": str(checkpoint),
        "config_sha256": hashlib.sha256(config.read_bytes()).hexdigest(),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }, indent=2))


def _prune_fold_checkpoints(work_dir: Path, keep: Path) -> int:
    """Keep the evaluated final checkpoint and remove redundant fold copies."""
    removed = 0
    keep_resolved = keep.resolve()
    for candidate in work_dir.glob("*.pth"):
        if candidate.resolve() != keep_resolved:
            candidate.unlink(missing_ok=True)
            removed += 1
    if removed:
        print(f"  [disk] removed {removed} redundant fold checkpoint(s)")
    (work_dir / 'last_checkpoint').write_text(str(keep_resolved), encoding='utf-8')
    return removed


def _run_training(
    config: Path, work_dir: Path, gpu_ids: list[str], resume: bool, launcher: Path
) -> None:
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_ids)
    environment["PYTHONPATH"] = str(launcher.parent)
    train_tool = _mmdet_tool("train")
    if len(gpu_ids) > 1:
        command = [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node", str(len(gpu_ids)),
            "--master_port", str(random.randint(20000, 29999)),
            str(launcher), train_tool, str(config), "--work-dir", str(work_dir),
            "--launcher", "pytorch",
        ]
    else:
        command = [
            sys.executable, str(launcher), train_tool, str(config),
            "--work-dir", str(work_dir)
        ]
    if resume:
        checkpoint = _best_checkpoint(work_dir)
        if checkpoint is None:
            raise FileNotFoundError(f'No checkpoint available to resume under {work_dir}')
        # CheckpointHook's pointer may refer to a pruned intermediate file.
        (work_dir / 'last_checkpoint').write_text(str(checkpoint.resolve()), encoding='utf-8')
        command.append("--resume")
    print(f"  [train] GPUs={gpu_ids}; {' '.join(command)}")
    _bounded_call(command, environment, float(environment.get('SESSION23_TRAIN_SECONDS', '7200')) + 180.)


def _write_prediction_config(
    train_config: Path, destination: Path, valid_json: Path, prefix: Path
) -> None:
    override = f'''

# Out-of-fold prediction override generated by SESSION 23.
# _delete_=True: fully replace training test_dataloader so no stale
# batch_sampler/filter_cfg/persistent_workers keys leak in.
test_dataloader = dict(
    _delete_=True,
    batch_size=1, num_workers=2, persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', data_root='',
        ann_file={str(valid_json)!r}, data_prefix=dict(img=''),
        metainfo=metainfo, test_mode=True, pipeline=test_pipeline,
    ),
)
test_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(valid_json)!r},
    metric=['bbox', 'segm'], format_only=True,
    outfile_prefix={str(prefix)!r},
)
test_cfg = dict(_delete_=True, type='TestLoop')
# Evaluation has no optimizer lifecycle; never invoke training-only hooks.
custom_hooks = []
load_from = None
resume = False
'''
    destination.write_text(train_config.read_text() + override)


def _run_prediction(
    config: Path, checkpoint: Path, work_dir: Path, gpu_id: str,
    launcher: Path,
) -> None:
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = gpu_id
    environment["PYTHONPATH"] = str(launcher.parent)
    _bounded_call(
        [
            sys.executable, str(launcher), _mmdet_tool("test"),
            str(config), str(checkpoint), "--work-dir", str(work_dir),
        ],
        environment,
        float(environment.get("SESSION23_DEADLINE", str(time.time() + 3600))) - time.time() - 600,
    )


# %% 6. COCO evaluation and out-of-fold aggregation
def _coco_evaluate(ground_truth: Path, predictions: Path, names: list[str]) -> dict:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    detections = json.loads(predictions.read_text())
    if not detections:
        return {
            "bbox_mAP50": 0.0,
            "segm_mAP50": 0.0,
            "segm_mAP50_95": 0.0,
            "per_class_mAP50": {name: 0.0 for name in names},
        }
    coco_gt = COCO(str(ground_truth))
    # Also accept fold JSON files generated by older Session 23 notebooks.
    coco_gt.dataset.setdefault("info", {})
    coco_dt = coco_gt.loadRes(str(predictions))

    bbox = COCOeval(coco_gt, coco_dt, "bbox")
    bbox.evaluate(); bbox.accumulate(); bbox.summarize()
    segmentation = COCOeval(coco_gt, coco_dt, "segm")
    segmentation.evaluate(); segmentation.accumulate(); segmentation.summarize()
    per_class = {}
    for category_id, name in enumerate(names, start=1):
        evaluator = COCOeval(coco_gt, coco_dt, "segm")
        evaluator.params.catIds = [category_id]
        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
        per_class[name] = float(evaluator.stats[1])
    return {
        "bbox_mAP50": float(bbox.stats[1]),
        "segm_mAP50": float(segmentation.stats[1]),
        "segm_mAP50_95": float(segmentation.stats[0]),
        "per_class_mAP50": per_class,
    }


def _merge_oof_predictions(files: list[Path], destination: Path) -> int:
    merged = []
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing out-of-fold prediction file: {path}")
        merged.extend(json.loads(path.read_text()))
    destination.write_text(json.dumps(merged))
    return len(merged)


def _parse_folds(value: str) -> list[int]:
    folds = sorted({int(part.strip()) for part in value.split(",") if part.strip()})
    if not folds or any(fold not in range(5) for fold in folds):
        raise ValueError("--folds must contain comma-separated values from 0 through 4")
    return folds


def _running_in_notebook_kernel() -> bool:
    """Recognize IPython's launcher without suppressing real CLI errors."""
    launcher = Path(sys.argv[0]).name if sys.argv else ""
    if launcher in {"colab_kernel_launcher.py", "ipykernel_launcher.py"}:
        return True
    try:
        shell = get_ipython()  # type: ignore[name-defined]
    except NameError:
        return False
    return shell is not None and hasattr(shell, "kernel")


# %% 7. End-to-end Session 23 run
def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 23: Cascade Mask R-CNN Swin-B DCNv2 five-fold CV"
    )
    parser.add_argument("--data-root", default="auto")
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--device", default="auto", help="auto, 0, or 0,1")
    parser.add_argument("--folds", default="0,1,2,3,4")
    parser.add_argument("--max-iters", type=int, default=6000,
                        help="Iterations per fold; benchmark QUICK before committing to five folds.")
    parser.add_argument("--hours", type=float, default=11.3)
    parser.add_argument("--image-size", type=int, default=640)
    parser.add_argument("--accumulation", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--skip-existing", action="store_true", default=True)
    parser.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--skip-train", action="store_true")
    # A pasted notebook cell inherits kernel flags such as ``-f connection.json``.
    # A genuine command-line invocation must still reject unknown arguments.
    if argv is None and _running_in_notebook_kernel():
        argv = []
    args = parser.parse_args(argv)

    if not 0.5 <= args.hours <= 11.3:
        raise ValueError("--hours must be between 0.5 and 11.3")
    if args.max_iters < 1:
        raise ValueError("--max-iters must be positive")
    if args.image_size < 384 or args.image_size % 32:
        raise ValueError("--image-size must be >=384 and divisible by 32")
    if args.accumulation < 1:
        raise ValueError("--accumulation must be positive")

    multiprocessing.freeze_support()
    started = float(os.environ.setdefault("SESSION23_STARTED", str(time.time())))
    print("=" * 76)
    print("SESSION 23 — Cascade Mask R-CNN Swin-B + real DCNv2 + five-fold CV")
    print(f"BUILD: {RUNTIME_BUILD_ID}")
    print(f"PYTHON: {sys.executable}")
    print("=" * 76)
    relaunched_exit_code = _ensure_dependencies(args)
    if relaunched_exit_code is not None:
        return relaunched_exit_code
    gpu_ids = _detect_gpu_ids(args.device)
    selected_folds = _parse_folds(args.folds)
    print(f"  [gpu] training devices={gpu_ids}; folds={selected_folds}")

    root = _find_dataset_root(args.data_root)
    names = _load_class_names(root / "data.yaml")
    train_items = _gather_split(root, "train")
    valid_items = _gather_split(root, "valid")
    test_items = _gather_split(root, "test")
    trainval = train_items + valid_items
    if len({item["key"] for item in trainval}) != len(trainval):
        raise RuntimeError("Duplicate image paths exist in the development pool")
    print(
        f"  [data] trainval pool: {len(train_items)} train + "
        f"{len(valid_items)} valid = {len(trainval)} total; "
        f"untouched test={len(test_items)}"
    )
    group_report = _assign_inferred_groups(root, trainval + test_items)
    has_group_manifest = (root / "metadata" / "manifest.csv").is_file()
    # Legacy fold files may split augmentations; only a reviewed group manifest
    # authorizes reuse. Otherwise reconstruct group folds from the raw images.
    predefined = (_load_predefined_folds(root, trainval, test_items)
                  if has_group_manifest else None)
    if predefined is None:
        fold_data, integrity = _generate_folds(
            trainval, len(names), 5, args.seed, test_items)
        integrity["group_manifest_present"] = has_group_manifest
        print("  [integrity] using inferred source/exact-pixel group folds")
    else:
        fold_data, integrity = predefined
        for fold in fold_data:
            if ({item["group_id"] for item in fold["train"]}
                    & {item["group_id"] for item in fold["valid"]}):
                raise RuntimeError(f"Fold {fold['fold']} separates related source images")
            if ({item["pixel_hash"] for item in fold["train"]}
                    & {item["pixel_hash"] for item in fold["valid"]}):
                raise RuntimeError(f"Fold {fold['fold']} separates exact pixel copies")
        print(
            "  [integrity] predefined folds validated; "
            f"group_disjoint_verified={integrity['group_disjoint_verified']}"
        )
    integrity.update(group_report)

    default_work = Path("/kaggle/working") / (
        f"{SESSION_NAME}_quick" if args.quick else SESSION_NAME
    )
    work = Path(args.work_dir).expanduser().resolve() if args.work_dir else default_work
    work.mkdir(parents=True, exist_ok=True)
    coco_dir = work / "coco"
    configs_dir = work / "configs"
    predictions_dir = work / "oof_predictions"
    for directory in (coco_dir, configs_dir, predictions_dir):
        directory.mkdir(parents=True, exist_ok=True)

    if args.quick:
        selected_folds = [selected_folds[0]]
        args.max_iters = min(args.max_iters, 20)
        print("  [quick] one fold, 48 train / 24 validation images, 20 iterations")

    selected_data = []
    for fold in selected_folds:
        current = dict(fold_data[fold])
        if args.quick:
            current["train"] = _balanced_subset(current["train"], 48, args.seed)
            current["valid"] = _balanced_subset(current["valid"], 24, args.seed + 1)
        selected_data.append(current)

    all_items_by_key = {
        item["key"]: item
        for current in selected_data
        for item in current["train"] + current["valid"]
    }
    image_ids = {
        key: index + 1 for index, key in enumerate(sorted(all_items_by_key))
    }
    oof_items_by_key = {
        item["key"]: item
        for current in selected_data
        for item in current["valid"]
    }
    oof_items = [oof_items_by_key[key] for key in sorted(oof_items_by_key)]
    oof_gt = coco_dir / "selected_oof_ground_truth.json"
    oof_conversion = _yolo_to_coco(oof_items, names, oof_gt, image_ids)

    # Build and validate the exact requested architecture before downloading
    # or training. The placeholder checkpoint path is not read during build.
    bootstrap_config = configs_dir / "architecture_preflight.py"
    shared_cache = (
        Path("/kaggle/working/session23_shared")
        if Path("/kaggle/working").is_dir() else work
    )
    safe_launcher = _write_empty_mask_guard(shared_cache)
    if str(shared_cache) not in sys.path:
        sys.path.insert(0, str(shared_cache))
    prepared_path = (
        shared_cache / "pretrained" / "cascade_swin_b_exactshape_v9.pth"
    )
    _write_training_config(
        bootstrap_config, oof_gt, oof_gt, work / "preflight", prepared_path,
        names, len(gpu_ids), args.max_iters, args.image_size,
        args.accumulation, args.seed,
    )
    compile(bootstrap_config.read_text(), str(bootstrap_config), "exec")
    print("  [preflight] building model and verifying Cascade/Swin-B/DCNv2 ...")
    _validate_model_build(bootstrap_config, safe_launcher)
    pretrained, adaptation_report = _download_and_prepare_checkpoint(
        shared_cache, bootstrap_config
    )
    summary = {
        "session": "SESSION_23",
        "runtime_build_id": RUNTIME_BUILD_ID,
        "model": "Cascade Mask R-CNN Swin-B with three DCNv2 mask heads",
        "method": "five-fold cross-validation with strict out-of-fold evaluation",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(root),
        "classes": names,
        "selected_folds": selected_folds,
        "dataset_integrity": integrity,
        "oof_conversion": oof_conversion,
        "checkpoint_adaptation": adaptation_report,
        "configuration": {
            "max_iters_per_fold": args.max_iters,
            "image_size": args.image_size,
            "gpus": gpu_ids,
            "batch_per_gpu": 1,
            "gradient_accumulation": args.accumulation,
            "quick": args.quick,
            "cascade_iou_thresholds": [0.5, 0.6, 0.7],
            "dcn_location": "four FCNMaskHead refinement convolutions per cascade stage (12 total)",
        },
        "fold_results": [],
        "oof": {},
        "limitations": [
            "The 0.75-0.81 mAP50 range in model_strategy_analysis.md is a planning "
            "estimate, not an empirical guarantee.",
            "The original test split is excluded from cross-validation; "
            "inferred source overlap can make it non-independent.",
            "Filename/pixel grouping is not verified patient identity or "
            "a substitute for clinician-reviewed detailed masks.",
            "Out-of-fold predictions use fixed final checkpoints; the held-out "
            "fold is not used for checkpoint selection.",
        ],
    }
    summary_path = work / "SESSION_23_results.json"

    def save_summary() -> None:
        summary_path.write_text(json.dumps(summary, indent=2, default=str))

    deadline = started + args.hours * 3600
    os.environ["SESSION23_DEADLINE"] = str(deadline)
    # Conservative initial allowance; updated from completed inference/scoring.
    evaluation_seconds = max(600., max(len(f['valid']) for f in selected_data) * 0.8 + 120.)
    shared_train_seconds = _fold_training_budget(deadline - time.time(), len(selected_data), evaluation_seconds)
    summary['configuration'].update(hours=args.hours, training_seconds_per_fold=shared_train_seconds,
        schedule='warmup and cosine decay by iteration or elapsed training time')
    save_summary()
    prediction_files = []
    for fold_position, current in enumerate(selected_data):
        fold = current["fold"]
        print("\n" + "-" * 76)
        print(f"FOLD {fold}: train={len(current['train'])}, valid={len(current['valid'])}")
        print("-" * 76)
        fold_coco = coco_dir / f"fold_{fold}"
        fold_config_dir = configs_dir / f"fold_{fold}"
        fold_work = work / "folds" / f"fold_{fold}"
        for directory in (fold_coco, fold_config_dir, fold_work):
            directory.mkdir(parents=True, exist_ok=True)
        train_json = fold_coco / "train.json"
        valid_json = fold_coco / "valid.json"
        train_report = _yolo_to_coco(current["train"], names, train_json, image_ids)
        valid_report = _yolo_to_coco(current["valid"], names, valid_json, image_ids)
        config = fold_config_dir / "cascade_maskrcnn_swin_b_dcn.py"
        _write_training_config(
            config, train_json, valid_json, fold_work, pretrained, names,
            len(gpu_ids), args.max_iters, args.image_size, args.accumulation,
            args.seed + fold,
        )
        compile(config.read_text(), str(config), "exec")

        checkpoint = _best_checkpoint(fold_work)
        completed_checkpoint = _completed_checkpoint(fold_work, config)
        if args.skip_train:
            if completed_checkpoint is None:
                raise FileNotFoundError(
                    f"--skip-train needs a config-matched completed checkpoint for fold {fold}"
                )
            checkpoint = completed_checkpoint
        elif args.skip_existing and completed_checkpoint is not None:
            checkpoint = completed_checkpoint
            print(f"  [train] reusing completed fold checkpoint: {checkpoint}")
        else:
            if checkpoint is not None and not args.resume:
                print(
                    "  [train] checkpoint exists without a matching completion "
                    "marker; starting this fold from iteration 0. Set RESUME=True "
                    "to resume an interrupted fold."
                )
            remaining_folds = len(selected_data) - fold_position
            allowance = min(shared_train_seconds, _fold_training_budget(
                deadline - time.time(), remaining_folds, evaluation_seconds))
            os.environ['SESSION23_TRAIN_SECONDS'] = str(allowance)
            print(f"  [budget] fold training allowance={allowance/3600:.2f} hours", flush=True)
            _run_training(config, fold_work, gpu_ids,
                          args.resume and checkpoint is not None, safe_launcher)
            checkpoint = _best_checkpoint(fold_work)
            if checkpoint is None:
                raise RuntimeError(f"Fold {fold} finished without a checkpoint")
            _mark_training_complete(fold_work, config, checkpoint)

        prediction_started = time.time()
        prediction_prefix = predictions_dir / f"fold_{fold}_valid"
        prediction_config = fold_config_dir / "predict_valid.py"
        _write_prediction_config(
            config, prediction_config, valid_json, prediction_prefix
        )
        print(f"  [oof] predicting fold {fold} validation with {checkpoint.name}")
        _run_prediction(prediction_config, checkpoint, fold_work,
                        gpu_ids[0], safe_launcher)
        prediction_json = Path(f"{prediction_prefix}.segm.json")
        if not prediction_json.is_file():
            raise FileNotFoundError(
                f"MMDetection did not create {prediction_json}"
            )
        metrics = _coco_evaluate(valid_json, prediction_json, names)
        evaluation_seconds = max(evaluation_seconds, (time.time() - prediction_started) * 1.25)
        prediction_files.append(prediction_json)
        summary["fold_results"].append({
            "fold": fold,
            "checkpoint": str(checkpoint),
            "training": json.loads((fold_work / "training_budget_report.json").read_text())
                if (fold_work / "training_budget_report.json").is_file() else None,
            "train_conversion": train_report,
            "valid_conversion": valid_report,
            "metrics": metrics,
        })
        _prune_fold_checkpoints(fold_work, checkpoint)
        save_summary()

    oof_predictions = predictions_dir / "SESSION_23_oof.segm.json"
    detection_count = _merge_oof_predictions(prediction_files, oof_predictions)
    oof_metrics = _coco_evaluate(oof_gt, oof_predictions, names)
    fold_scores = [
        result["metrics"]["segm_mAP50"] for result in summary["fold_results"]
    ]
    import statistics
    summary["oof"] = {
        "evaluated_images": len(oof_items),
        "detections": detection_count,
        "metrics": oof_metrics,
        "fold_mean_segm_mAP50": float(statistics.mean(fold_scores)),
        "fold_std_segm_mAP50": (
            float(statistics.pstdev(fold_scores)) if len(fold_scores) > 1 else 0.0
        ),
        "complete_five_fold_oof": selected_folds == [0, 1, 2, 3, 4] and not args.quick,
    }
    summary["elapsed_hours"] = (time.time() - started) / 3600.0
    save_summary()

    archive = Path("/kaggle/working/SESSION_23_V10_results.zip")
    if not Path("/kaggle/working").is_dir():
        archive = work / "SESSION_23_V10_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output_zip:
        output_zip.write(summary_path, arcname=summary_path.name)
        output_zip.write(oof_gt, arcname=oof_gt.name)
        output_zip.write(oof_predictions, arcname=oof_predictions.name)
        adaptation_path = pretrained.parent / "checkpoint_adaptation.json"
        output_zip.write(adaptation_path, arcname=adaptation_path.name)
        for current, prediction in zip(selected_data, prediction_files):
            fold = current["fold"]
            config = configs_dir / f"fold_{fold}" / "cascade_maskrcnn_swin_b_dcn.py"
            output_zip.write(config, arcname=f"fold_{fold}/{config.name}")
            output_zip.write(prediction, arcname=f"fold_{fold}/{prediction.name}")

    print("\n" + "=" * 76)
    print("SESSION 23 COMPLETE")
    print("=" * 76)
    print(
        f"OOF segm mAP50={oof_metrics['segm_mAP50']:.4f}; "
        f"mAP50-95={oof_metrics['segm_mAP50_95']:.4f}; "
        f"bbox mAP50={oof_metrics['bbox_mAP50']:.4f}"
    )
    print("Per-class OOF segmentation mAP50:")
    for name, value in oof_metrics["per_class_mAP50"].items():
        print(f"  {name}: {value:.4f}")
    print(f"Results JSON: {summary_path}")
    print(f"Results ZIP : {archive}")
    print(f"Checkpoints : {work / 'folds'}  (download separately)")
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    if _running_in_notebook_kernel():
        status = main([])
        if status:
            raise RuntimeError(f"Session 23 failed with exit status {status}")
    else:
        raise SystemExit(main())
